# Реализация Логистической Регрессии с нуля (NumPy)

В этом ноутбуке представлена реализация алгоритма логистической регрессии. Главная цель проекта - показать понимание внутренних механизмов алгоритмов машинного обучения (градиентный спуск, функции активации, регуляризация).

**Что реализовано в классе `MyLogisticRegression`:**
1. Пакетный градиентный спуск (Mini-Batch Gradient Descent) с использованием `yield`-генератора.
2. Бинарная кросс-энтропия (Log-Loss) в качестве функции потерь.
3. L2-регуляризация весов (Ridge) для предотвращения переобучения.
4. Векторизованные математические операции для максимальной скорости работы.
5. Инференс-методы в стиле sklearn (`predict` и `predict_proba`).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

In [ ]:
class MyLogisticRegression(object):
  def __init__(self, lr=0.01, epochs=10, penalty: str='l2', C=1.0):
      self.lr = lr
      self.epochs = epochs
      self.penalty = penalty
      self.C = C

      self.weights = None

      self.losses = []

  def sigmoid(self, x: np.ndarray) -> np.ndarray:
    return 1 / (1 + np.exp(-x))

  def batch_generator(self, X, y, batch_size):
    assert len(X) == len(y)

    perm = np.random.permutation(len(X))

    for batch_start in range(0, len(X), batch_size):
      batch_id = perm[batch_start: batch_start + batch_size]
      yield(X[batch_id],y[batch_id])

  def prediction_int(self, X):
    predict = self.sigmoid(np.dot(X, self.weights))
    return predict

  def loss_(self, y, y_pred):
    y_pred = np.clip(y_pred, 1e-10, 1 - 1e-10)
    return -np.mean(y * np.log(y_pred) + (1 - y) * np.log(1 - y_pred)) + (1 / (2 * self.C)) * np.sum(self.weights[1:]**2)

  def get_grad(self, X_batch, y_batch, predictions):
    reg_grad = np.copy(self.weights)
    reg_grad[0] = 0
    grad = X_batch.T @ (predictions - y_batch) / len(y_batch) + (1 / self.C) * reg_grad
    return grad

  def fit(self, X, y, batch_size=64) -> None:
    n_objects, n_features  = X.shape
    if self.weights is None:
      self.weights = np.random.randn(n_features + 1)

    X_train = np.concatenate((np.ones((n_objects, 1)), X), axis=1)

    for i in range(self.epochs):
      for X_batch, y_batch in self.batch_generator(X_train, y, batch_size):
         predictions = self.prediction_int(X_batch)
         loss = self.loss_(y_batch, predictions)
         self.losses.append(loss)
         grad = self.get_grad(X_batch, y_batch, predictions)
         self.weights -= grad * self.lr

  def predict_proba(self, X):
      n_objects = X.shape[0]
      X_with_bias = np.concatenate((np.ones((n_objects, 1)), X), axis=1)
      return self.prediction_int(X_with_bias)

  def predict(self, X, threshold=0.5):
      proba = self.predict_proba(X)
      return (proba >= threshold).astype(int)

In [ ]:
X, y = make_classification(n_samples=1000, n_features=5, n_informative=4,
                           n_redundant=0, random_state=42)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = MyLogisticRegression(lr=0.05, epochs=50, C=1.0)
model.fit(X_train_scaled, y_train, batch_size=32)

y_pred = model.predict(X_test_scaled)

accuracy = accuracy_score(y_test, y_pred)
print(f"Точность собвственной модели лог. регрессии на тестовой выборке: accuracy = {accuracy:.4f}")

plt.figure(figsize=(10, 6))
plt.plot(model.losses, color='blue', alpha=0.7)
plt.title("График обучения: Падение Loss (Log-Loss + L2)")
plt.xlabel("Итерации")
plt.ylabel("Значение Loss")
plt.grid(True)
plt.show()

Сравнение с моделью LogisticRegression из scikit-learn

In [ ]:
from sklearn.linear_model import LogisticRegression

sklearn_model = LogisticRegression(penalty='l2', C=1.0, random_state=42)
sklearn_model.fit(X_train_scaled, y_train)

y_pred_sklearn = sklearn_model.predict(X_test_scaled)

accuracy_sklearn = accuracy_score(y_test, y_pred_sklearn)
print(f"Точность моей модели: accuracy = {accuracy:.4f}")
print(f"Точность модели scikit-learn: accuracy = {accuracy_sklearn:.4f}")

print("\n--- Глубокая проверка: Сравнение весов ---")
print("Веса моей модели:")
print(f"Bias: {model.weights[0]:.4f}")
print(f"Коэффициенты: {model.weights[1:].round(4)}")

print("\nВеса модели scikit-learn:")
print(f"Bias: {sklearn_model.intercept_[0]:.4f}")
print(f"Коэффициенты: {sklearn_model.coef_[0].round(4)}")

### Выводы и результаты бенчмаркинга

На основе проведенного эксперимента можно сделать следующие выводы:

1. **Сходимость алгоритма:** График `Loss` демонстрирует плавное экспоненциальное затухание. Это доказывает, что математика градиентов, усреднение по размеру батча и шаг обучения работают корректно.
2. **Метрики качества:** Точность (accuracy) кастомной модели на тестовой выборке практически идентична результату эталонной модели `LogisticRegression` из `scikit-learn` (+-0.68 / 0/7 соотвественно). Алгоритм успешно улавливает линейные закономерности в данных.
3. **Анализ весов (Коэффициентов):** Значения весов и смещения (bias) имеют тот же знак и очень близкий порядок величин по сравнению с весами `sklearn`.

**Итог:** Модель полностью готова, математически обоснована и показывает метрики уровня промышленных библиотек.